# כוונון עדין (Fine-Tuning) למודל Gemma-2-270M לזיהוי פרטי שירים בעברית

מחברת זו מדגימה כיצד לבצע כוונון עדין (Fine-Tuning) למודל `gemma-2-270m-it` על מנת לחלץ מידע מובנה (שם זמר, שיר, אלבום וכו') משמות של שירים בעברית. 

**הכלים שנשתמש בהם:**
- **פלטפורמה:** Kaggle Notebooks (עם GPU T4)
- **מודל בסיס:** `google/gemma-2-270m-it`
- **מערך נתונים:** `NHLOCAL/SingNER` מ-Hugging Face
- **ספריית אימון:** `Unsloth` לאימון מהיר ויעיל בזיכרון בשיטת QLoRA.

## שלב 1: התקנת ספריות והגדרות ראשוניות

נתקין את הספריות הנדרשות. Unsloth דורשת התקנה מיוחדת בסביבת Kaggle כדי להבטיח תאימות מלאה וביצועים מקסימליים.

In [ ]:
%%capture
# התקנה מיוחדת עבור Unsloth בסביבת Kaggle
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

# התקנת ספריות נוספות
!pip install --no-deps "xformers<0.0.26"
!pip install --no-deps "transformers>=4.38.0"
!pip install --no-deps "datasets>=2.16.0"
!pip install --no-deps "accelerate>=0.26.0"
!pip install --no-deps "trl>=0.8.3"
!pip install --no-deps "peft>=0.10.0"

## שלב 2: ייבוא ספריות והגדרת קונפיגורציה

נייבא את כל הספריות הנדרשות ונגדיר את כל הפרמטרים המרכזיים במילון קונפיגורציה אחד (`CONFIG`) כדי לשמור על סדר ונוחות.

In [ ]:
import torch
import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig

# מילון קונפיגורציה מרכזי
CONFIG = {
    "model_name": "google/gemma-2-270m-it",
    "dataset_name": "NHLOCAL/SingNER",
    "max_seq_length": 256,  # אורך רצף קצר מספיק עבור שמות שירים
    
    # הגדרות LoRA (ערכים קטנים מתאימים למשימה פשוטה)
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_bias": "none",
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    
    # הגדרות אימון
    "output_dir": "./gemma-ner-finetuned",
    "num_train_epochs": 2,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "logging_steps": 10,
    "optim": "paged_adamw_8bit",
    "seed": 42
}

print("התצורה הוגדרה בהצלחה!")

## שלב 3: טעינת והמרת מערך הנתונים

זהו השלב הקריטי בו אנו הופכים את מערך הנתונים מפורמט NER לפורמט של הוראה-תשובה (Instruction-Response) שהמודל יכול ללמוד ממנו.

הפורמט החדש ייראה כך:
```
### הוראה:
חלץ את ישויות המוזיקה מהטקסט הבא והחזר אותן בפורמט JSON.

### קלט:
{טקסט השיר המקורי}

### פלט:
{JSON עם הישויות שחולצו}
```

In [ ]:
def format_for_llm_training(example):
    """ 
    ממיר דוגמה בפורמט NER לפורמט הוראה-תשובה עבור אימון LLM.
    הפלט הוא מחרוזת JSON עם שמות הישויות, לא המיקומים שלהן.
    """
    text = example['text']
    entities = example['entities']
    
    # יצירת מילון לאחסון הישויות שחולצו
    extracted_entities = {
        "SINGER": [],
        "SONG": [],
        "ALBUM": [],
        "GENRE": [],
        "MISC": []
    }
    
    for entity in entities:
        label = entity['label']
        # חילוץ הטקסט של הישות לפי מיקומי ההתחלה והסיום
        entity_text = text[entity['start']:entity['end']]
        if label in extracted_entities:
            extracted_entities[label].append(entity_text)
            
    # סינון מפתחות ריקים מהמילון
    final_entities = {k: v for k, v in extracted_entities.items() if v}
    
    # המרת המילון למחרוזת JSON מסודרת
    # ensure_ascii=False נדרש כדי לשמור על תווים בעברית
    output_json = json.dumps(final_entities, ensure_ascii=False, indent=2)
    
    # יצירת התבנית המלאה לאימון
    instruction_prompt = f"""### הוראה:
חלץ את ישויות המוזיקה מהטקסט הבא והחזר אותן בפורמט JSON.

### קלט:
{text}

### פלט:
{output_json}"""
    
    return {"text": instruction_prompt}

# טעינת מערך הנתונים מ-Hugging Face
dataset = load_dataset(CONFIG['dataset_name'])

# החלת פונקציית הפורמט על כל הדוגמאות
formatted_dataset = dataset.map(format_for_llm_training)

# הדפסת דוגמה אחת כדי לוודא שההתאמה בוצעה כראוי
print("--- דוגמה לאחר המרה ---")
print(formatted_dataset['train'][0]['text'])

## שלב 4: טעינת המודל והוספת מתאם LoRA

כעת נטען את מודל ה-Gemma באמצעות Unsloth, מה שיבצע אופטימיזציות אוטומטיות וטעינה ב-4bit. לאחר מכן, נוסיף לו מתאם LoRA (PEFT) כדי לאמן רק אחוז קטן מהמשקולות, מה שחוסך המון זיכרון וזמן.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CONFIG['model_name'],
    max_seq_length = CONFIG['max_seq_length'],
    dtype = None,  # יזהה אוטומטית (bfloat16 ב-T4)
    load_in_4bit = True,
)

# הוספת מתאם LoRA למודל
model = FastLanguageModel.get_peft_model(
    model,
    r = CONFIG['lora_r'],
    lora_alpha = CONFIG['lora_alpha'],
    lora_dropout = CONFIG['lora_dropout'],
    bias = CONFIG['lora_bias'],
    target_modules = CONFIG['lora_target_modules'],
)

print("המודל והמתאם LoRA נטענו בהצלחה.")
print(f"מספר פרמטרים לאימון: {model.get_nb_trainable_parameters():,}")

## שלב 5: כוונון עדין (Fine-Tuning)

נגדיר את ארגומנטי האימון ונשתמש ב-`SFTTrainer` של ספריית `trl` כדי לבצע את הכוונון העדין. התהליך צפוי להיות מהיר בזכות כל האופטימיזציות שביצענו.

In [ ]:
training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    logging_steps=CONFIG['logging_steps'],
    optim=CONFIG['optim'],
    seed=CONFIG['seed'],
    fp16 = not torch.cuda.is_bf16_supported(), # שימוש ב-fp16 אם bfloat16 לא נתמך
    bf16 = torch.cuda.is_bf16_supported(),
    save_strategy="epoch",
    report_to="none" # ניתן לשנות ל-"wandb" למעקב מתקדם
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset['train'],
    dataset_text_field="text",
    max_seq_length=CONFIG['max_seq_length'],
    args=training_args,
    packing=True, # אריזת דוגמאות קצרות יחד לניצול יעיל של ה-GPU
)

print("מתחיל אימון...")
trainer.train()
print("האימון הסתיים בהצלחה!")

## שלב 6: בדיקת המודל המאומן (Inference)

לאחר סיום האימון, נבדוק את המודל על מספר דוגמאות חדשות כדי לראות אם הוא למד לחלץ את המידע ולהחזיר JSON תקין.

In [ ]:
# הכנת המודל ל-Inference מהיר
FastLanguageModel.for_inference(model)

# דוגמאות לבדיקה
test_examples = [
    "יוני בלוך - אחריות",
    "התקווה 6 - הכי ישראלי (מתוך האלבום הכי ישראלי)",
    "כהן@מושון - מה קורה לי",
    "דודו טסה והכוויתים - ליו ולב",
    "שיר רוק של משינה בשם רכבת לילה"
]

for example in test_examples:
    prompt = f"""### הוראה:
חלץ את ישויות המוזיקה מהטקסט הבא והחזר אותן בפורמט JSON.

### קלט:
{example}

### פלט:"""
    
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    response = tokenizer.batch_decode(outputs)
    
    print(f"--- בדיקה עבור: '{example}' ---")
    # ניקוי הפלט כדי להציג רק את התשובה של המודל
    cleaned_response = response[0].split("### פלט:")[1].replace('<eos>', '').strip()
    print(cleaned_response)
    print("\n")

## שלב 7: שמירת המודל (אופציונלי)

אם אתה מרוצה מהתוצאות, תוכל לשמור את מתאם ה-LoRA המאומן לשימוש עתידי.

In [ ]:
adapter_save_path = "gemma_270m_singner_lora"
model.save_pretrained(adapter_save_path) # שומר רק את מתאם ה-LoRA הקטן
tokenizer.save_pretrained(adapter_save_path)

print(f"המתאם נשמר בנתיב: {adapter_save_path}")

# ניתן גם להעלות אותו ל-Hugging Face Hub
# from huggingface_hub import login
# login()
# model.push_to_hub("your_username/gemma_270m_singner_lora", token = True)